In [1]:
# Cell 1: Install packages (OFFLINE MODE)

# Replace this with your copied path!
wheels_path = "/kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels" 

# Install everything strictly from your local folder
!pip install -U --no-index --find-links $wheels_path transformers peft bitsandbytes trl accelerate datasets

print("✅ Offline installation complete!")

Looking in links: /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/transformers-5.4.0-py3-none-any.whl
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/trl-1.0.0-py3-none-any.whl
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/accelerate-1.13.0-py3-none-any.whl
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/datasets-4.8.4-py3-none-any.whl
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/huggingface_hub-1.8.0-py3-none-any.whl (from transformers)
Processing /kaggle/input/datasets/azizkarray/my-qwen-2-5-wheels/my_wheels/hf_xet-1.4.2-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (from huggingface-hub<2.0,>=1.5.0->transformers)
  Attempting uninstall: hf-xet
    Fou

In [2]:
# Cell 2: Imports and Setup
import os
import torch
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ✅ Set seed for exact reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

# ✅ Hardware optimizations
# Enable TF32 for faster matrix multiplications (works on Ampere GPUs like Kaggle's A100/T4x2)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ✅ Use bfloat16 if the Kaggle GPU supports it, otherwise fallback to float16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"🚀 Environment ready! Using compute dtype: {compute_dtype}")

🚀 Environment ready! Using compute dtype: torch.bfloat16


In [3]:
# Cell 3: Data Loading & Analysis
import pandas as pd

# Load the dataset
# Update the path if Kaggle mounted it differently (e.g., /kaggle/input/llm-classification-finetuning/train.csv)
TRAIN_PATH = "/kaggle/input/competitions/llm-classification-finetuning/train.csv" 
df = pd.read_csv(TRAIN_PATH)

print(f"✅ Total training samples loaded: {len(df)}\n")

# --- 1. Label Distribution Check ---
print("📊 Label Distribution:")
# The dataset has binary columns for the winners
label_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
distribution = df[label_cols].sum()
print(distribution)
print("\nPercentage:")
print((distribution / len(df) * 100).round(2).astype(str) + '%')

# --- 2. Text Length Analysis (Approximate words) ---
print("\n📏 Text Length Analysis (Word count approximations):")
df['prompt_len'] = df['prompt'].apply(lambda x: len(str(x).split()))
df['res_a_len'] = df['response_a'].apply(lambda x: len(str(x).split()))
df['res_b_len'] = df['response_b'].apply(lambda x: len(str(x).split()))

# Looking at the 90th percentile gives us a realistic ceiling for truncation
print("90th Percentile Lengths (Words):")
print(f"Prompt:     {df['prompt_len'].quantile(0.90):.0f} words")
print(f"Response A: {df['res_a_len'].quantile(0.90):.0f} words")
print(f"Response B: {df['res_b_len'].quantile(0.90):.0f} words")

✅ Total training samples loaded: 57477

📊 Label Distribution:
winner_model_a    20064
winner_model_b    19652
winner_tie        17761
dtype: int64

Percentage:
winner_model_a    34.91%
winner_model_b    34.19%
winner_tie         30.9%
dtype: object

📏 Text Length Analysis (Word count approximations):
90th Percentile Lengths (Words):
Prompt:     117 words
Response A: 406 words
Response B: 406 words


In [4]:
# Cell 4: Label Processing

def get_single_token_label(row):
    if row['winner_model_a'] == 1:
        return "A"
    elif row['winner_model_b'] == 1:
        return "B"
    else:
        return "C"  # C represents the "Tie" class

# Apply the function to create our single-token target
df['target'] = df.apply(get_single_token_label, axis=1)

# Verify the mapping
print("✅ Labels processed into single tokens (A, B, C):")
print(df['target'].value_counts())

✅ Labels processed into single tokens (A, B, C):
target
A    20064
B    19652
C    17761
Name: count, dtype: int64


In [5]:
# Cell 5: Swap Augmentation (50% random swap)
import numpy as np

# Create a boolean mask for approximately 50% of the rows
swap_mask = np.random.rand(len(df)) > 0.5

print(f"🔄 Swapping A and B for {swap_mask.sum()} rows ({(swap_mask.sum()/len(df))*100:.1f}%)")

# --- 1. Swap the text in the response columns ---
temp_res_a = df.loc[swap_mask, 'response_a'].copy()
df.loc[swap_mask, 'response_a'] = df.loc[swap_mask, 'response_b']
df.loc[swap_mask, 'response_b'] = temp_res_a

# --- 2. Swap the model names to keep the dataset honest ---
temp_mod_a = df.loc[swap_mask, 'model_a'].copy()
df.loc[swap_mask, 'model_a'] = df.loc[swap_mask, 'model_b']
df.loc[swap_mask, 'model_b'] = temp_mod_a

# --- 3. Swap the targets ---
# 'A' becomes 'B', 'B' becomes 'A', 'C' (Tie) stays 'C'
target_swap_map = {'A': 'B', 'B': 'A', 'C': 'C'}
df.loc[swap_mask, 'target'] = df.loc[swap_mask, 'target'].map(target_swap_map)

print("✅ Swap augmentation complete!")
print("\n📊 New label distribution after swap:")
print(df['target'].value_counts())

🔄 Swapping A and B for 28753 rows (50.0%)
✅ Swap augmentation complete!

📊 New label distribution after swap:
target
A    19918
B    19798
C    17761
Name: count, dtype: int64


In [6]:
# Cell 6: Prompt Formatting
import ast

def parse_and_join(text):
    """Safely evaluates the stringified list and joins multi-turn dialogues."""
    try:
        # FIX: Clean the string to prevent Python SyntaxWarnings
        clean_text = text.replace('\\/', '/')
        
        # Convert string '["turn 1", "turn 2"]' into actual Python list
        parsed_list = ast.literal_eval(clean_text)
        # Join multiple turns with a double newline for readability
        return "\n\n".join(parsed_list)
    except Exception as e:
        # Fallback just in case a row is malformed
        return str(text)

def create_prompt(row):
    """Wraps the parsed text into the instruction format."""
    prompt = parse_and_join(row['prompt'])
    res_a = parse_and_join(row['response_a'])
    res_b = parse_and_join(row['response_b'])
    
    # We leave the final answer blank because we will extract the logits 
    # for the very next token the model predicts after '### Answer:\n'
    template = f"""### Instruction:
Given a prompt and two responses, determine which response is better.

### Prompt:
{prompt}

### Response A:
{res_a}

### Response B:
{res_b}

### Answer:
"""
    return template

print("⚙️ Formatting prompts...")
df['formatted_prompt'] = df.apply(create_prompt, axis=1)

print("✅ Formatting complete! Here is a sneak peek of the first sample:\n")
print("-" * 50)
print(df['formatted_prompt'].iloc[0][:600] + "\n\n... [TRUNCATED] ...")
print("-" * 50)

⚙️ Formatting prompts...
✅ Formatting complete! Here is a sneak peek of the first sample:

--------------------------------------------------
### Instruction:
Given a prompt and two responses, determine which response is better.

### Prompt:
Is it morally right to try to have a certain percentage of females on managerial positions?

OK, does pineapple belong on a pizza? Relax and give me fun answer.

### Response A:
The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in favor:**

1. **Correc

... [TRUNCATED] ...
--------------------------------------------------


In [7]:
# Cell 7: Smart Truncation (Head + Tail Strategy)

def smart_truncate(text, max_words):
    """Truncates text by keeping the head (30%) and tail (70%)."""
    words = str(text).split()
    if len(words) <= max_words:
        return text
    
    # Calculate head and tail lengths
    head_len = int(max_words * 0.30)
    tail_len = max_words - head_len
    
    # Join head and tail with a marker in the middle
    truncated_text = " ".join(words[:head_len]) + "\n\n... [TRUNCATED] ...\n\n" + " ".join(words[-tail_len:])
    return truncated_text

def create_truncated_prompt(row):
    """Parses, truncates, and wraps the text in the instruction format."""
    # 1. Parse strings (using the ast logic we built in Step 5)
    prompt = parse_and_join(row['prompt'])
    res_a = parse_and_join(row['response_a'])
    res_b = parse_and_join(row['response_b'])
    
    # 2. Smart Truncate BEFORE template injection
    prompt_trunc = smart_truncate(prompt, max_words=450)
    res_a_trunc = smart_truncate(res_a, max_words=525)
    res_b_trunc = smart_truncate(res_b, max_words=525)
    
    # 3. Inject into template
    template = f"""### Instruction:
Given a prompt and two responses, determine which response is better.

### Prompt:
{prompt_trunc}

### Response A:
{res_a_trunc}

### Response B:
{res_b_trunc}

### Answer:
"""
    return template

print("✂️ Applying Smart Truncation and re-formatting...")
df['formatted_prompt'] = df.apply(create_truncated_prompt, axis=1)

# Let's verify our max lengths
df['final_word_count'] = df['formatted_prompt'].apply(lambda x: len(x.split()))
print(f"✅ Truncation complete!")
print(f"📊 Maximum word count in any final prompt: {df['final_word_count'].max()} words")

✂️ Applying Smart Truncation and re-formatting...
✅ Truncation complete!
📊 Maximum word count in any final prompt: 1532 words


In [8]:
# Cell 8: Hugging Face Dataset Conversion & Train/Val Split
from datasets import Dataset

print("🧹 Scrubbing broken Unicode/surrogates from the text...")
# This forces the text into valid UTF-8, replacing broken characters safely
df['formatted_prompt'] = df['formatted_prompt'].apply(
    lambda x: str(x).encode('utf-8', errors='replace').decode('utf-8')
)

print("📦 Converting pandas DataFrame to Hugging Face Dataset...")

# 1. Select only the columns we actually need for training to save memory
cols_to_keep = ['formatted_prompt', 'target']
hf_dataset = Dataset.from_pandas(df[cols_to_keep])

# 2. Split the dataset (90% Train, 10% Validation)
# We use a fixed seed (42) so our split is reproducible across runs
print("🪓 Splitting into Train and Validation sets (90/10)...")
dataset_dict = hf_dataset.train_test_split(test_size=0.1, seed=42)

# Rename 'test' to 'eval' to match standard Hugging Face Trainer naming conventions
dataset_dict['eval'] = dataset_dict.pop('test')

print("\n✅ Dataset preparation complete!")
print("-" * 30)
print(f"📚 Training samples:   {len(dataset_dict['train'])}")
print(f"🎯 Validation samples: {len(dataset_dict['eval'])}")
print("-" * 30)

# Let's peek at the final structure
print("\nFinal Dataset Structure:")
print(dataset_dict)

🧹 Scrubbing broken Unicode/surrogates from the text...
📦 Converting pandas DataFrame to Hugging Face Dataset...
🪓 Splitting into Train and Validation sets (90/10)...

✅ Dataset preparation complete!
------------------------------
📚 Training samples:   51729
🎯 Validation samples: 5748
------------------------------

Final Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['formatted_prompt', 'target'],
        num_rows: 51729
    })
    eval: Dataset({
        features: ['formatted_prompt', 'target'],
        num_rows: 5748
    })
})


In [9]:
# Cell 9: Tokenization (Speed Optimized)
from transformers import AutoTokenizer

model_id = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1" 
print(f"🤖 Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

label_map = {'A': 0, 'B': 1, 'C': 2}

def tokenize_function(examples):
    # Dropping max_length to 512 for a massive speedup!
    tokenized = tokenizer(
        examples['formatted_prompt'],
        truncation=True,
        max_length=512, 
    )
    tokenized['labels'] = [label_map[label] for label in examples['target']]
    return tokenized

print("⏳ Re-tokenizing the dataset...")
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True, remove_columns=['formatted_prompt', 'target'])
print("✅ Tokenization complete!")

🤖 Loading tokenizer for /kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1...
⏳ Re-tokenizing the dataset...


Map:   0%|          | 0/51729 [00:00<?, ? examples/s]

Map:   0%|          | 0/5748 [00:00<?, ? examples/s]

✅ Tokenization complete!


In [10]:
# Cell 11: Loading the Model & PEFT/LoRA Setup
import torch
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1" 

# 1. Quantization Configuration (Squishing the model to fit in Kaggle RAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Speeds up computation on modern GPUs
)

print(f"🧠 Loading {model_id} base model in 4-bit mode...")
# 2. Load the base model with a Classification Head
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=3,
    quantization_config=bnb_config,
    device_map="auto" # Automatically figures out the best way to use your GPU
)

# Crucial: Tell the model what our padding token is
model.config.pad_token_id = tokenizer.pad_token_id

# 3. Prepare the model for parameter-efficient 4-bit training
model = prepare_model_for_kbit_training(model)

print("🛠️ Configuring LoRA adapters...")
# 4. LoRA Configuration
lora_config = LoraConfig(
    r=16, # The "rank" (size) of the adapters
    lora_alpha=32, # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Targeting the attention mechanisms
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS" # Explicitly telling LoRA this is Sequence Classification
)

# Apply the LoRA adapters to the model
model = get_peft_model(model, lora_config)

print("\n✅ Model and Adapters loaded successfully!")
print("-" * 50)
# This will show you exactly how few parameters we are actually training
model.print_trainable_parameters()
print("-" * 50)

🧠 Loading /kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1 base model in 4-bit mode...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen2.5/transformers/7b-instruct/1
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🛠️ Configuring LoRA adapters...

✅ Model and Adapters loaded successfully!
--------------------------------------------------
trainable params: 10,103,296 || all params: 7,080,733,184 || trainable%: 0.1427
--------------------------------------------------


In [11]:
# Cell 12: Training Arguments and Trainer Setup (Speed Optimized)
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

print("⚙️ Setting up Training Arguments...")

training_args = TrainingArguments(
    output_dir="./qwen-chatbot-arena-model",
    learning_rate=2e-4,              
    per_device_train_batch_size=4,   
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   
    num_train_epochs=1,              
    eval_strategy="steps",           
    eval_steps=100,                  # Lowered to 100 since our dataset is smaller
    save_strategy="steps",
    save_steps=100,                  
    logging_steps=50,               
    fp16=True,                       
    optim="paged_adamw_8bit",        
    report_to="none"                 
)

print("🧩 Initializing Data Collator...")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("🧑‍🏫 Initializing the Trainer with a 10k Subset...")

trainer = Trainer(
    model=model,
    args=training_args,
    # Slicing the dataset to 10,000 for training and 1,000 for evaluation
    train_dataset=tokenized_datasets["train"].select(range(10000)),
    eval_dataset=tokenized_datasets["eval"].select(range(1000)),
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("✅ Trainer is completely ready!")

⚙️ Setting up Training Arguments...
🧩 Initializing Data Collator...
🧑‍🏫 Initializing the Trainer with a 10k Subset...
✅ Trainer is completely ready!


In [12]:
# Cell 13: The Training Loop & Saving
import warnings
warnings.filterwarnings("ignore") # Suppress minor huggingface warnings during output

print("🚀 Blast off! Starting the training loop...")
# This single line handles the entire forward pass, backward pass, and optimization!
trainer.train()

print("\n💾 Saving the final model and tokenizer...")
# Save the trained LoRA adapters
trainer.save_model("./qwen-chatbot-arena-final")
# Save the tokenizer so we can use it during inference later
tokenizer.save_pretrained("./qwen-chatbot-arena-final")

print("🎉 Training complete! You have successfully fine-tuned an LLM!")

🚀 Blast off! Starting the training loop...


`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Accuracy
100,9.809285,1.854360,0.404000
200,5.305836,1.125234,0.374000
300,4.521381,1.146123,0.406000
400,4.417423,1.064593,0.439000
500,4.319545,1.046929,0.454000
600,4.098604,1.016490,0.477000
625,4.098604,1.018752,0.483000



💾 Saving the final model and tokenizer...
🎉 Training complete! You have successfully fine-tuned an LLM!


In [13]:
# Cell 14: Inference Function (The "Judge" in Action)
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

def judge_conversation(prompt, response_a, response_b):
    # 1. Load the path where we saved our masterpiece
    model_path = "./qwen-chatbot-arena-final"
    base_model_id = "Qwen/Qwen2.5-7B-Instruct"
    
    print("🧠 Loading the Judge...")
    # Load tokenizer and base model again (or use existing if in same session)
    test_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # 2. Format the prompt exactly like we did during training
    formatted_input = f"Instruction: {prompt}\n\nModel A: {response_a}\n\nModel B: {response_b}"
    
    # 3. Tokenize and move to GPU
    inputs = test_tokenizer(formatted_input, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    
    # 4. Get the prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Convert logits to probabilities
        probabilities = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        prediction = torch.argmax(logits, dim=-1).item()
    
    # 5. Map back to labels
    label_lookup = {0: 'A', 1: 'B', 2: 'Tie (C)'}
    winner = label_lookup[prediction]
    
    print(f"\n--- 🤖 JUDGE'S VERDICT ---")
    print(f"Winner: Model {winner}")
    print(f"Confidence: {probabilities[prediction]:.2%}")
    print(f"Breakdown: A: {probabilities[0]:.1%}, B: {probabilities[1]:.1%}, Tie: {probabilities[2]:.1%}")

# --- EXAMPLE TEST CASE ---
test_prompt = "Explain quantum physics to a five-year-old."
test_a = "Quantum physics is the study of very small things like atoms and particles. It's like magic but real!"
test_b = "Imagine the world is made of tiny LEGO blocks that can be in two places at once until you look at them."

# Call the judge
judge_conversation(test_prompt, test_a, test_b)

🧠 Loading the Judge...

--- 🤖 JUDGE'S VERDICT ---
Winner: Model B
Confidence: 48.32%
Breakdown: A: 30.2%, B: 48.3%, Tie: 21.5%


In [14]:
# Cell 15: OOM-Safe Official Submission Generation (FIXED)
import pandas as pd
import torch

print("📂 Loading test data...")
# Make sure this path exactly matches your competition's data folder!
test_df = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")

# ---------------------------------------------------------
# 🚨 THE FIX IS HERE 🚨
# Instead of writing a new simplified lambda function, we 
# reuse the exact function from Cell 7 that includes your 
# parsing, smart truncation, and '###' headers!
# ---------------------------------------------------------
print("⚙️ Formatting test prompts exactly like training data...")
test_df['formatted_prompt'] = test_df.apply(create_truncated_prompt, axis=1)

model.eval() # Set model to evaluation mode
all_probs = []

print("🤖 Generating predictions in safe batches...")

# 2. Process in small chunks to prevent GPU memory crashes!
batch_size = 4  # Keep this small (2-8) for a 7B model
for i in range(0, len(test_df), batch_size):
    batch_prompts = test_df['formatted_prompt'].iloc[i : i + batch_size].tolist()
    
    # Tokenize just this small batch
    inputs = tokenizer(
        batch_prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    ).to("cuda")
    
    # Run inference on the batch
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)

# 3. Create the Submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': [p[0] for p in all_probs],
    'winner_model_b': [p[1] for p in all_probs],
    'winner_tie': [p[2] for p in all_probs]
})

# 4. Save to CSV
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv created successfully with PERFECT formatting!")
display(submission.head())

📂 Loading test data...
⚙️ Formatting test prompts exactly like training data...
🤖 Generating predictions in safe batches...
✅ submission.csv created successfully with PERFECT formatting!


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.200301,0.348974,0.450725
1,211333,0.234341,0.392735,0.372925
2,1233961,0.406270,0.341779,0.251951
